# Ingest sprints.json file
1. Read the all the files from the sprints folder using spark dataframe reader API
1. Define and enforce schema 
1. Add Metadata Columns 
    - Source File
    - Ingestion Timestamp
1. Write to bronze delta table    

> Note: JSON is in multi line format

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
# Define source_file and table_name
source_file = f"{landing_folder_path}/sprints"
table_name = f"{catalog_name}.{bronze_schema}.sprints"

#### Step 1 - Read the JSON file using the dataframe reader API

In [0]:
# Define the schema
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType, DateType

sprints_schema = StructType([
    StructField("date", StringType()),
    StructField("raceName", StringType()),
    StructField("round", IntegerType()),
    StructField("season", IntegerType()),
    StructField("url", StringType()),
    StructField("constructorId", StringType()),
    StructField("driverId", StringType()),
    StructField("grid", IntegerType()),
    StructField("laps", IntegerType()),
    StructField("number", IntegerType()),
    StructField("points", FloatType()),
    StructField("position", IntegerType()),
    StructField("positionText", StringType()),
    StructField("status", StringType())
])


In [0]:
# Read data from the sprints file
sprints_df = (
    spark.read
       .format('json')
       .schema(sprints_schema)
       .option('mode', 'FAILFAST')
       .option('multiLine', True)
       .load(source_file)
)

In [0]:
display(sprints_df)

#### Step 2 - Add Metadata Columns
- Source File
- Ingestion Timestamp

In [0]:
sprints_final_df = add_ingestion_metadata(sprints_df)

#### Step 3 - Write to bronze delta table

In [0]:
(
    sprints_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(table_name)
)

In [0]:
display(spark.table(table_name))